# 02 · Feature extraction
Load only frozen split manifests. Build the 84-value tabular representation and full-clip log-mel power caches. Do not inspect or plot validation/test content; only report counts and shapes.

In [1]:
from pathlib import Path
import sys,json,hashlib,numpy as np,pandas as pd,yaml
sys.path.insert(0,str(Path('../src').resolve()))
from dronedet import features
from dronedet.config import load_config
from dronedet.data_preparation import locate_dataset_root
PROJECT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
CFG=yaml.safe_load((PROJECT/'config.yaml').read_text(encoding='utf-8'))
profile='colab' if 'google.colab' in sys.modules else 'local'; P={k:Path(v) for k,v in CFG['paths'][profile].items()}
def safe(path):
 path=Path(path).resolve(); raw=P['raw_root'].resolve()
 if path==raw or raw in path.parents: raise PermissionError(path)
 path.parent.mkdir(parents=True,exist_ok=True); return path
fe=load_config(PROJECT/'config.yaml'); root=locate_dataset_root(P['raw_root'])
hashes=json.loads((PROJECT/'results/tables/manifest_hashes.json').read_text(encoding='utf-8'))
for s in ('train','val','test'):
 assert hashlib.sha256((PROJECT/'data/processed/splits'/f'{s}.csv').read_bytes()).hexdigest()==hashes[s], f'{s} split changed after freeze'
print(fe.validate_filterbank(), 'n_win=',fe.n_win,'win_samples=',fe.win_samples)

{'minimum_nonzero_bins': 3, 'filters_with_fewer_than_3_bins': 0, 'lowest_half_width_hz': 28.638780933028375, 'fft_bin_spacing_hz': 15.625, 'margin_bins': 1.832881979713816} n_win= 28 win_samples= 7936


D:\Work\ML Course Project\Functional Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
summaries={}; train_cache=train_index=None
for split in ('train','val','test'):
 df=pd.read_csv(PROJECT/'data/processed/splits'/f'{split}.csv',keep_default_na=False)
 cache,index,X=features.build_split_cache(df,fe,root)
 assert len(index)==len(df)==len(X) and X.shape[1]==84 and np.isfinite(X).all()
 np.save(safe(PROJECT/'data/processed/spectrograms'/f'mel_power_{split}.npy'),cache)
 index.to_csv(safe(PROJECT/'data/processed/spectrograms'/f'index_{split}.csv'),index=False,lineterminator='\n')
 np.savez_compressed(safe(PROJECT/'data/processed/features_tabular'/f'{split}.npz'),X=X,y=df.label.to_numpy(np.int8),clip_id=df.clip_id.to_numpy(str),group=df.group.to_numpy(str),feature_names=np.asarray(features.feature_names()),silent=index.silent.to_numpy(),near_silent=index.near_silent.to_numpy(),crop_all_zero=index.crop_all_zero.to_numpy(),partial_zero=df.partial_zero.to_numpy(),clipped_flag=df.clipped_flag.to_numpy(),dc_flag=df.dc_flag.to_numpy())
 summaries[split]={'clips':len(df),'mel_shape':list(cache.shape),'features_shape':list(X.shape),'silent':int(index.silent.sum()),'resampled':int((index.length_status!='unchanged').sum()),'crop_all_zero':int(index.crop_all_zero.sum())}
 if split=='train': train_cache,train_index=cache,index
 print(split,summaries[split])
assert sum(v['clips'] for v in summaries.values())==10623
mu,sigma,frame_count=features.global_stats(train_cache,train_index,fe)
assert mu.shape==(64,) and sigma.shape==(64,) and np.isfinite(mu).all() and np.isfinite(sigma).all() and (sigma>0).all()
np.savez_compressed(safe(PROJECT/'data/processed/spectrograms/norm_stats_global.npz'),mean=mu.astype(np.float32),std=sigma.astype(np.float32),frame_count=np.int64(frame_count),clips_used=np.int64(train_index.stats_eligible.sum()),clips_excluded=np.int64((~train_index.stats_eligible).sum()))
print('train-only global stats:',frame_count,'frames;',int(train_index.stats_eligible.sum()),'eligible clips')
print('Decision log draft: Tabular descriptors use 19 MFCCs without c0, HTK mel centres from edge frequencies, rolloff 0.85, population standard deviation, and power=1 flatness; mel cache is full-clip and all centering uses frame-aligned offsets.')

train {'clips': 7607, 'mel_shape': [64, 457897], 'features_shape': [7607, 84], 'silent': 9, 'resampled': 0, 'crop_all_zero': 69}


val {'clips': 1510, 'mel_shape': [64, 91761], 'features_shape': [1510, 84], 'silent': 0, 'resampled': 0, 'crop_all_zero': 8}


test {'clips': 1506, 'mel_shape': [64, 91494], 'features_shape': [1506, 84], 'silent': 0, 'resampled': 0, 'crop_all_zero': 19}


train-only global stats: 452765 frames; 7519 eligible clips
Decision log draft: Tabular descriptors use 19 MFCCs without c0, HTK mel centres from edge frequencies, rolloff 0.85, population standard deviation, and power=1 flatness; mel cache is full-clip and all centering uses frame-aligned offsets.


In [ ]:
# Check report-filterbank parity using the owner's implementation; save once and never regenerate a golden.
from dronedet.config import FrontEndConfig
from dronedet.dsp import mel_filterbank
reference=FrontEndConfig.reference()
ours=mel_filterbank(reference.sr,reference.n_fft,reference.n_mels,reference.fmin,reference.fmax)
lib=__import__('librosa').filters.mel(sr=reference.sr,n_fft=reference.n_fft,n_mels=reference.n_mels,fmin=reference.fmin,fmax=reference.fmax,htk=True,norm='slaney',dtype=np.float32)
support=np.array_equal(ours>0,lib>0)
relative=float(np.max(np.abs(ours[lib>0]-lib[lib>0])/np.abs(lib[lib>0])))
assert support and relative<=1e-6
golden_path=PROJECT/'data/processed/golden/report_filterbank.npy'
if golden_path.exists(): assert np.array_equal(np.load(golden_path,allow_pickle=False),ours)
else:
 with golden_path.open('xb') as stream: np.save(stream,ours,allow_pickle=False)
print('report filterbank parity: identical support; max relative error',relative)
assert sum(v['clips'] for v in summaries.values())==10623
assert all(v['features_shape'][1]==84 for v in summaries.values())
for split in ('train','val','test'):
 df=pd.read_csv(PROJECT/'data/processed/splits'/f'{split}.csv',keep_default_na=False)
 idx=pd.read_csv(PROJECT/'data/processed/spectrograms'/f'index_{split}.csv',keep_default_na=False)
 assert len(df)==len(idx) and idx.clip_id.equals(df.clip_id)
print('feature gates:',pd.read_csv(PROJECT/'data/interim/features_gates.csv').groupby('ok').size().to_dict())
print('F13: saved 16/48 kHz synthetic goldens and report filterbank both pass parity.')
print('Decision log draft: Tabular descriptors use 19 MFCCs without c0, HTK mel centres from edge frequencies, rolloff 0.85, population standard deviation, and power=1 flatness; mel cache is full-clip and all centering uses frame-aligned offsets.')